# Week 1 Assignment: Project Charter
## STAT 390 | Spring 2026

---

## 1. Research Question

> **After the first set of an ATP Grand Slam match is complete, what is the optimal way to combine a player's pre-match historical profile with their live first-set performance to predict who wins the match — and does that optimal weighting vary by surface?**

Formally, given a model of the form:

$$P(A\ \text{wins}) = f\bigl(w \cdot \mathbf{x}_{\text{first-set}} + (1-w) \cdot \mathbf{x}_{\text{pre-match}}\bigr)$$

find the value of $w^* \in [0,1]$ that minimizes prediction error, and test whether $w^*_{\text{clay}} \neq w^*_{\text{grass}} \neq w^*_{\text{hard}}$.

---
## 2. One-Page Project Charter

| | |
|---|---|
| **Project** | Predicting ATP Grand Slam Match Outcomes Using Pre-Match History and Live First-Set Performance |
| **Course** | STAT 390, Spring 2026 |
| **Problem** | Tennis prediction models either use only historical data (pre-match) or only live data (in-match). Neither alone is optimal. The first set already contains strong signal about that day's performance, but history provides context the first set cannot. The right answer is a blend — but what blend? |
| **Data** | Jeff Sackmann's `tennis_slam_pointbypoint` (point-by-point, 2011–2023) + `tennis_atp` (pre-match metadata). Final dataset: **3,638 matches × 52 features**, ATP men's Grand Slams only. |
| **Method** | Four modeling stages: (1) pre-match logistic baseline, (2) first-set logistic baseline, (3) weighted combination with grid search over $w$, (4) tree-based models (Random Forest, XGBoost) with SHAP. |
| **AI Agent** | Built entirely inside Claude Code following Karpathy's AutoResearch loop. Agent observes data/output, forms hypotheses, writes and runs code, interprets results, and proposes next steps autonomously. Human reviews at four checkpoints. |
| **Timeline** | Weeks 1–2: pipeline + EDA · Weeks 3–4: baseline models · Weeks 5–6: combination + SHAP · Weeks 7–8: surface analysis + writeup |
| **Deliverable** | A fully reproducible pipeline and a written answer to the research question with quantified accuracy, calibration, and surface-stratified weights. |

---
## 3. Success Criterion

**The combined model (Stage 3) must outperform both the pre-match-only and first-set-only baselines on the held-out 2022–2023 test set, with at least +2 percentage points accuracy improvement over the better of the two baselines, and a Brier score ≤ 0.18.**

Secondary criteria:
- Calibration curve within ±5% of the diagonal at every decile
- A clear, interpretable statement of the optimal weight $w^*$ (e.g. "first-set features should receive 65% of the weight")
- Bootstrap confidence interval showing whether $w^*$ differs significantly across surfaces
- Fully reproducible: entire pipeline runs with `python src/01_download_data.py && ... && python src/05_model.py`

---
## 4. AutoResearch Workflow Diagram

The project is built using Karpathy's **AutoResearch** framework: an AI agent that operates autonomously inside a Observe → Think → Act loop, with the human providing the goal and reviewing at milestones.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import matplotlib.patheffects as pe
import numpy as np

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 9)
ax.axis('off')
fig.patch.set_facecolor('#F8F9FA')
ax.set_facecolor('#F8F9FA')

def box(ax, x, y, w, h, text, color, textsize=9.5, textcolor='white', bold=False, subtext=None):
    patch = FancyBboxPatch((x - w/2, y - h/2), w, h,
                            boxstyle="round,pad=0.08", facecolor=color,
                            edgecolor='white', linewidth=1.5, zorder=3)
    ax.add_patch(patch)
    weight = 'bold' if bold else 'normal'
    ax.text(x, y if subtext is None else y + 0.13, text,
            ha='center', va='center', fontsize=textsize, color=textcolor,
            fontweight=weight, zorder=4, wrap=True)
    if subtext:
        ax.text(x, y - 0.22, subtext, ha='center', va='center',
                fontsize=7.5, color=textcolor, alpha=0.85, zorder=4, style='italic')

def arrow(ax, x1, y1, x2, y2, label='', color='#555', curve=0.0):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8,
                                connectionstyle=f'arc3,rad={curve}'))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx + 0.05, my, label, fontsize=7.5, color=color, ha='left', va='center', style='italic')

# ── Title
ax.text(7, 8.55, 'AutoResearch Workflow: ATP Match Prediction',
        ha='center', va='center', fontsize=13, fontweight='bold', color='#222')
ax.text(7, 8.15, 'Human sets goal  ·  Agent operates loop  ·  Human reviews checkpoints',
        ha='center', va='center', fontsize=9, color='#666')

# ── Human Goal box (top left)
box(ax, 2.2, 7.0, 3.4, 0.9, 'HUMAN: Set Goal',
    '#1A237E', textsize=10, bold=True,
    subtext='"Find optimal weight w between\npre-match & first-set features"')

# ── Agent loop circle — 4 nodes
# Center of loop: (8.5, 4.5)
cx, cy, r = 8.5, 4.3, 2.3
angles = [90, 0, 270, 180]   # top, right, bottom, left
loop_labels  = ['OBSERVE', 'THINK', 'ACT', 'OBSERVE']
loop_sub     = ['Read data & output', 'Form hypothesis / plan', 'Write & run code', 'Interpret results']
loop_colors  = ['#1565C0', '#6A1B9A', '#1B5E20', '#00838F']
node_coords  = []
for ang, lbl, sub, col in zip(angles, loop_labels, loop_sub, loop_colors):
    rad = np.radians(ang)
    nx, ny = cx + r * np.cos(rad), cy + r * np.sin(rad)
    node_coords.append((nx, ny))
    box(ax, nx, ny, 2.4, 0.82, lbl, col, textsize=10, bold=True, subtext=sub)

# Arrows around the loop
loop_arcs = [(0,1,0.3),(1,2,0.3),(2,3,0.3),(3,0,0.3)]
for i,j,c in loop_arcs:
    arrow(ax, node_coords[i][0], node_coords[i][1],
              node_coords[j][0], node_coords[j][1], curve=c, color='#444')

# Loop label in center
ax.text(cx, cy, 'AGENT\nLOOP', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#888',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#F8F9FA', edgecolor='#CCC', linewidth=1))

# ── Arrow from Human Goal to loop
arrow(ax, 3.9, 7.0, node_coords[0][0]-0.3, node_coords[0][1]+0.42, color='#1A237E')

# ── Human Checkpoints (right side)
chk_color = '#B71C1C'
checkpoints = [
    (12.0, 6.8, 'CP 1: Dataset selection', 'Confirm slam_pbp over ATP-only'),
    (12.0, 5.5, 'CP 2: Clean data review', 'Schema + null profile approved'),
    (12.0, 4.2, 'CP 3: Model results', 'Review accuracy / AUC, decide next stage'),
    (12.0, 2.9, 'CP 4: Final report', 'Review interpretation before submission'),
]
for cx2, cy2, lbl, sub in checkpoints:
    box(ax, cx2, cy2, 3.4, 0.78, lbl, chk_color, textsize=8.5, bold=True, subtext=sub)
    arrow(ax, node_coords[2][0]+1.0, node_coords[2][1], cx2-1.7, cy2, color=chk_color, curve=0)

# ── Iteration labels (bottom)
iters = [
    (2.5, 1.8, '#1565C0', 'Iter 1–2\nData pipeline + EDA'),
    (5.5, 1.8, '#6A1B9A', 'Iter 3–4\nBaseline models'),
    (8.5, 1.8, '#1B5E20', 'Iter 5–6\nWeight search + SHAP'),
    (11.5, 1.8, '#00838F', 'Iter 7–8\nSurface analysis + writeup'),
]
for ix, iy, ic, il in iters:
    box(ax, ix, iy, 2.6, 0.9, il, ic, textsize=8.5, bold=False)

ax.text(7, 1.1, '8-Week Timeline', ha='center', fontsize=9, color='#555', style='italic')

plt.tight_layout()
plt.savefig('../data/plots/autoresearch_workflow.png', dpi=150, bbox_inches='tight',
            facecolor='#F8F9FA')
plt.show()
print('Saved to data/plots/autoresearch_workflow.png')

**How the loop maps to this project:**

| Iteration | Observe | Think | Act | Result |
|---|---|---|---|---|
| 1 | No data exists | Best source is Jeff Sackmann's slam_pbp | Write `01_download_data.py` | 49 tournament files downloaded |
| 2 | Raw points: 1 row/point | Need 1 row/match with set-1 aggregates | Write `02_build_firstset_features.py` with multi-format parsing | 10,377 matches processed |
| 3 | Join rate only 30.8% | Order-independent name key needed | Switch to sorted pair key in `03_join_and_clean.py` | 3,638 clean matches |
| 4 | Clean dataset ready | Understand feature correlations before modeling | Write `04_explore_clean_data.py` | Key predictors identified; avg_rally dropped |

---
## 5. Risk List

| # | Risk | Likelihood | Impact | Mitigation |
|---|---|---|---|---|
| R1 | **Data format changes** mid-tournament era (IBM → Infosys) silently corrupt feature values | High (already occurred) | High | Multi-format detection with sanity checks (set-1 win rate should be ~77–81%) |
| R2 | **Player name mismatches** between slam_pbp and ATP metadata cause low join rate | High (already occurred) | High | Sorted name-pair key; manual inspection of unmatched rows |
| R3 | **Target leakage** from first-set features that implicitly encode the final score | Medium | Critical | Strict feature audit: only aggregate stats through set 1, point 1 onward; no set 2+ data |
| R4 | **Class imbalance** if random A/B assignment fails | Low | Medium | Verify `A_won.mean()` ≈ 0.50 after every pipeline run |
| R5 | **Serve stat nulls** (16–18%) biased toward early years cause imputation to introduce era-based confounding | Medium | Medium | Impute per-surface-per-year median; add missingness indicator; compare models with/without early years |
| R6 | **Overfitting** in Stage 4 tree models on 3,638 rows with 52 features | Medium | Medium | 5-fold CV; hold out 2022–2023 as a true test set never seen during tuning |
| R7 | **Scope creep** (adding H2H records, serve direction, etc.) prevents completion in 8 weeks | Medium | Medium | Strict scope freeze after Week 4; stretch goals only if Stages 1–3 are done |
| R8 | **Result is trivial** (first-set win is so dominant that w*≈1.0 and pre-match adds nothing) | Low | Low | Still publishable: quantifying *how little* history matters after set 1 is a meaningful finding |

---
## 6. Repository Structure (First Draft)

```
tennis-match-prediction/
│
├── README.md                          # Project overview, dataset summary, modeling roadmap
├── requirements.txt                   # Python dependencies
├── .gitignore                         # Excludes raw/interim/clean data (too large)
│
├── src/                               # Reproducible pipeline scripts (run in order)
│   ├── 01_download_data.py            # Download 49 tournament files from JeffSackmann repos
│   ├── 02_build_firstset_features.py  # Aggregate point-by-point → first-set stats per match
│   ├── 03_join_and_clean.py           # Join with ATP metadata, clean, balance target
│   ├── 04_explore_clean_data.py       # EDA: correlations, distributions, surface splits
│   └── 05_model.py                    # (upcoming) Stages 1–4 modeling pipeline
│
├── notebooks/                         # Jupyter notebooks: analysis + weekly check-ins
│   ├── week1_charter.ipynb            # ← this file
│   ├── week2_checkin.ipynb            # Pipeline complete, EDA findings
│   └── week3_baseline_models.ipynb    # (upcoming)
│
├── data/
│   ├── raw/                           # Raw tournament CSV files (gitignored, ~200MB)
│   │   └── slam_pbp/                  # 49 × 2 = 98 files (matches + points per tournament)
│   ├── interim/
│   │   └── firstset_features.csv      # 10,377 matches with set-1 aggregates (gitignored)
│   ├── clean/
│   │   └── tennis_model_ready.csv     # 3,638 × 52 model-ready dataset (gitignored)
│   └── plots/                         # EDA figures (committed — small PNGs)
│       ├── 01_overview.png
│       ├── 02_correlation_heatmap.png
│       ├── 03_firstset_stats_by_outcome.png
│       ├── 04_feature_distributions.png
│       └── autoresearch_workflow.png
│
└── reports/
    └── project_proposal.txt           # Full narrative proposal
```

**Design decisions:**
- `src/` scripts are numbered and self-contained — run them sequentially to rebuild the dataset from scratch
- `notebooks/` are for analysis and reporting only — they read from `data/`, never write to it
- Large data files are gitignored; only plots (small PNGs) and final scripts are tracked
- Weekly check-in notebooks accumulate week by week, providing a running project log